# 1500 Random Networks -- Pipeline Robustness / Singularity Check

Scaled up from the original 150-case run (10x) -- the point of this notebook isn't
illustration, it's a statistical robustness claim: across a diverse sample of random
(N, a) stick networks, is the reduced Laplacian matrix M ever singular after
percolation + pruning + boundary merge? At 150 cases, only 57 were actually solved
(percolating + surviving pruning), which is a thin evidence base for "never singular."
Scaling to 1500 cases (~570 expected solved, at the same ~38% rate) makes that claim
meaningfully stronger, plus gives tighter effective-resistance / percolation-rate /
pruning-rate statistics across the (N, a) range.

`PLOT_EACH_CASE` is `False` here (was `True` for the 150-case version) -- per-case
before/after-pruning and before/after-merge figures make sense to eyeball for 150
cases, not for 1500+ (huge notebook, wasted render time on a headless cluster node
with nobody watching). `analyze_case(case_id, results, examples)` in Section 8 still
works for any individual case afterward, on demand -- nothing about that path changed.


## 1. Setup and Imports

In [1]:
# Environment and path setup (same pattern used in PiaNo.ipynb)
import math
import sys
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    utils_path = "/content/drive/MyDrive/ColabProject/utils"
    print("Environment: Google Colab")
except ImportError:
    utils_path = os.path.abspath(os.path.join(os.getcwd(), "..", "utils"))
    print("Environment: Local")

if utils_path not in sys.path:
    sys.path.append(utils_path)

print(f"Utils path: {utils_path}")
print(f"Files available: {os.listdir(utils_path)}")


Environment: Local
Utils path: <project_root>/utils
Files available: ['random_network_batch.py', 'self_loop_utils.py', 'node_loop_test_runner.py', '.DS_Store', 'region_checks.py', 'path_analysis.py', 'test_merge.py', 'rebuild_experiments.py', 'multi_terminal_solver.py', 'impedance_calculator.py', 'self_loops.py', 'pipeline_utils.py', 'pdf_generator.py', 'network_visualization.py', 'pruning.py', 'utility_reset.py', '__pycache__', 'edge_discovery.py', 'point_generator_3d.py', 'path_pairs_of_lines vs  path_pairs_of_edges.txt', 'test_percolation_3d.py', 'intersection_utils.py', 'plotting.py', 'admittance_checks.py', 'loop_topology_utils.py', 'network_visualization_3d.py', 'node_loop_visualization.py', 'plotting_old.py', 'point_generator.py', 'filtered_paths.py', 'pipeline_utils_src.py', 'nextSteps.txt', 'intersection_utils_3d.py', 'experiment_manager.py', 'boundary_utils.py', 'laplacian_utils.py', 'result_exporter.py', 'pipeline_utils_3d.py', 'test_networks.py']


In [2]:
# All the pipeline logic (percolation -> pruning -> boundary merge ->
# impedance, plus the batch runner, summary and plots) lives in
# utils/random_network_batch.py -- this notebook is just a thin driver.
from random_network_batch import (
    generate_diversified_configs, run_batch, build_results_dataframe,
    summarize_batch, plot_percolation_overview, analyze_case,
)

import pandas as pd
import matplotlib.pyplot as plt

print("All modules imported successfully!")

All modules imported successfully!


---
## Experiment tracking

In [3]:
from experiment_manager import setup_experiment

exp_manager = setup_experiment()

EXPERIMENT CONFIGURATION
  Environment: Local
  Experiment #: 789
  Timestamp: 20260817_144545
  Results directory: <project_root>/utils/../experiments/exp_789



## 2. Diversified Parameter Grid (1500 Random Networks)


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# 1500 DIVERSIFIED (N, a) CONFIGURATIONS
# ═══════════════════════════════════════════════════════════════════════
N_CASES = 1500
N_MIN, N_MAX = 8, 90          # number of sticks
A_MIN, A_MAX = 0.15, 0.80     # stick length (geometric limit is sqrt(2) ~= 1.4142)
K_OHM = 1000.0                 # resistance per segment (same convention as PiaNo.ipynb)

configs = generate_diversified_configs(
    N_CASES, N_MIN, N_MAX, A_MIN, A_MAX, param_seed=2025, case_seed_base=10_000
)

print(f"Generated {len(configs)} diversified (N, a) configurations")
print(f"  N range: {min(c['N'] for c in configs)} - {max(c['N'] for c in configs)}")
print(f"  a range: {min(c['a'] for c in configs):.3f} - {max(c['a'] for c in configs):.3f}")
print("\nFirst 10 configs:")
for c in configs[:10]:
    print(f"  case {c['case_id']:3d}: N={c['N']:3d}  a={c['a']:.3f}  seed={c['seed']}")


Generated 1500 diversified (N, a) configurations
  N range: 8 - 90
  a range: 0.150 - 0.800

First 10 configs:
  case   0: N= 79  a=0.204  seed=10000
  case   1: N= 69  a=0.765  seed=10001
  case   2: N= 75  a=0.150  seed=10002
  case   3: N= 56  a=0.520  seed=10003
  case   4: N= 16  a=0.399  seed=10004
  case   5: N= 20  a=0.178  seed=10005
  case   6: N= 35  a=0.788  seed=10006
  case   7: N= 20  a=0.417  seed=10007
  case   8: N= 86  a=0.168  seed=10008
  case   9: N= 73  a=0.193  seed=10009


## 3. Full Pipeline for a Single Network



## 4. Run All 150 Cases




In [5]:
# False for the 1500-case run: per-case plotting is what made 150 cases
# slow/heavy already, and would be both wasted (headless cluster node,
# nobody watching plt.show()) and enormous (1500x before/after-pruning +
# before/after-merge figures) at this scale. Set True for a small local run
# where you actually want to eyeball every case as it goes.
PLOT_EACH_CASE = False

import json

cluster_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "cluster"))
cache_dir = os.path.join(cluster_dir, "random_networks_results")
N_TASKS = 20  # must match run_random_networks_task.py
chunk_files = [os.path.join(cache_dir, f"chunk_{i:02d}.json") for i in range(N_TASKS)]
use_cache = all(os.path.exists(p) for p in chunk_files)

if use_cache:
    print(f"Found all {N_TASKS} chunk results cached in {cache_dir} "
          f"(from the parallel SLURM array run) -- loading instead of recomputing.")
    results = []
    for p in chunk_files:
        with open(p) as f:
            results.extend(json.load(f))
    # The array script only saves the flat per-case summary rows, not the raw
    # per-case data (points/intersections/merged network) -- Section 8's
    # analyze_case() demo won't have anything to show for these cache-loaded
    # results, but that doesn't affect the Section 5/6 statistical summary,
    # which only needs the rows.
    examples = {}
    print(f"Loaded {len(results)} cached results.")
else:
    if os.path.isdir(cache_dir):
        n_found = sum(os.path.exists(p) for p in chunk_files)
        print(f"Cache dir {cache_dir} exists but is incomplete ({n_found}/{N_TASKS} chunks) -- recomputing all.")
    results, examples = run_batch(configs, k=K_OHM, plot=PLOT_EACH_CASE, verbose=False)


Found all 20 chunk results cached in <project_root>/cluster/random_networks_results (from the parallel SLURM array run) -- loading instead of recomputing.


Loaded 1500 cached results.


## 5. Summary Table

In [6]:
df = build_results_dataframe(results)

pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 160)

df[['case_id', 'N', 'a', 'n_intersections', 'has_percolation', 'needed_pruning',
    'nodes_removed', 'edges_removed', 'proceeds_after_pruning',
    'effective_resistance', 'singular_label', 'condition_number', 'status']]


,case_id,N,a,n_intersections,has_percolation,needed_pruning,nodes_removed,edges_removed,proceeds_after_pruning,effective_resistance,singular_label,condition_number,status
0,0,79,0.204,89,False,False,0,0,False,inf,N/A,NaN,no_percolating_path
1,1,69,0.765,1293,True,False,0,0,True,1820.330324,no,2882.397245,ok
2,2,75,0.150,41,False,False,0,0,False,inf,N/A,NaN,no_percolating_path
3,3,56,0.520,396,False,False,0,0,False,inf,N/A,NaN,no_boundary_nodes
4,4,16,0.399,13,False,False,0,0,False,inf,N/A,NaN,no_boundary_nodes
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,1495,44,0.501,191,True,False,0,0,True,2641.031762,no,720.158203,ok
1496,1496,79,0.240,131,False,False,0,0,False,inf,N/A,NaN,no_percolating_path
1497,1497,47,0.320,89,False,False,0,0,False,inf,N/A,NaN,no_percolating_path
1498,1498,34,0.216,13,False,False,0,0,False,inf,N/A,NaN,no_boundary_nodes


## 6. Aggregate Statistics and Singularity Verdict



In [7]:
stats, singular_df = summarize_batch(df)

if len(singular_df):
    display(singular_df)


SUMMARY OVER 1500 RANDOM NETWORKS
Total networks generated:                  1500
Percolating (source-drain path exists):     645 (43.0%)
Required pruning/removal of dead-ends:      141 (9.4%)
Successfully solved for V and R:             645 (43.0%)

Singularity of the reduced Laplacian matrix M (answered for every example):
  Singular   (cond(M) > 1e10):    0
  Non-singular (invertible):      645
  N/A (no linear system solved):  855

Effective resistance statistics (solved cases only):
  min    = 995.838 Ohm
  median = 2350.707 Ohm
  max    = 22124.078 Ohm

No singular matrices were found among the solved networks.


## 7. Overview Plots

In [8]:
fig = plot_percolation_overview(df, K_OHM)
plt.show()
exp_manager.save_figure(fig, "percolation_overview.png")

  [OK] Figure saved: percolation_overview.png


'<project_root>/utils/../experiments/exp_789/percolation_overview.png'

## 8. On-Demand Case Analysis (Visual + Circuit)


In [9]:
# analyze_case(case_id, results, examples, ...) lives in
# utils/random_network_batch.py -- callable here for any of the 150 cases.

# Quick demo: first pruned case
pruned_ids = df.loc[df['needed_pruning'], 'case_id'].tolist()
if pruned_ids:
    analyze_case(pruned_ids[0], results, examples)


Case 21: no data available (status is likely 'generation_failed').


---
## 9. Save consolidated summary

In [10]:
import json

# df.to_json/json.loads round-trip sanitizes numpy types (np.float64, np.bool_,
# etc.) into plain JSON-serializable Python types -- simpler than casting each
# column by hand.
results_json_safe = json.loads(df.to_json(orient="records"))

summary = {
    "experiment": exp_manager.get_info(),
    "config": {"N_CASES": N_CASES, "N_MIN": N_MIN, "N_MAX": N_MAX,
               "A_MIN": A_MIN, "A_MAX": A_MAX, "K_OHM": K_OHM,
               "plot_each_case": PLOT_EACH_CASE},
    "stats": stats,
    "results": results_json_safe,
}

exp_manager.save_json(summary, "random_networks_batch_summary.json")

print(f"\nAll results saved to: {exp_manager.experiment_dir}")
for f in sorted(os.listdir(exp_manager.experiment_dir)):
    print(f"  - {f}")

  [OK] JSON saved: random_networks_batch_summary.json

All results saved to: <project_root>/utils/../experiments/exp_789
  - percolation_overview.png
  - random_networks_batch_summary.json
